In [8]:
from dotenv import load_dotenv

load_dotenv()

True

In [9]:
from typing import Annotated, Optional, TypedDict
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    document: Optional[str]

In [10]:
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import InjectedState
from langgraph.types import Command
from langchain_core.messages import ToolMessage

@tool
def read_document(
    document: Annotated[str, InjectedState("document")]  # only injects the document field
) -> str:
    """Read the current document content

    Args:
        document (Annotated[str, InjectedState): the document state

    Returns:
        str: the content of the document
    """
    return document

@tool
def update(
    new_content: str,
    state: Annotated[AgentState, InjectedState]
) -> Command:
    """Updates the document with the provided content.

    Args:
        new_content (str): the complete new content for the document
    """
    
    print(new_content)
    return Command(
        update={
            "document": new_content,  # writes directly to state
            "messages": [
                ToolMessage(
                    content=f"Document updated successfully",
                    tool_call_id=state["messages"][-1].tool_calls[0]["id"]
                )
            ]
        }
    )

@tool
def save(filename: str, document: Annotated[str, InjectedState('document')]) -> str:
    """Save the current document to a text file and finish the process

    Args:
        filename (str): Name of the text file
        document (Annotated[str, InjectedState): the document state

    Returns:
        str: flag showing status of the saved document
    """
    
    if not filename.endswith('.txt'):
        filename = f'{filename}.txt'
        
    try:
        with open(filename, 'w') as f:
            f.write(document)
        return f"Document saved successfully to {filename}"
    except Exception as e:
        return f"Failed to save the document. {str(e)}"
    

tools = [read_document, update, save ]

model = ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite-preview').bind_tools(tools)

In [11]:
from typing import Literal

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage

def agent(state: AgentState) -> dict:
    system_prompt = SystemMessage(content=f"""
    You are Drafter, a helpful writing assistant.

    - When the user asks for edits, rewrite the ENTIRE document incorporating their changes.
    - Pass the COMPLETE rewritten document to the 'update' tool, not just the changed parts.
    - Never pass just a summary or description of changes — always the full content.

    Current document: {state['document']}
    """)
    if not state['messages']:
        # AI speaks first, greeting the user
        opening_message = AIMessage(content="I'm ready to help you update a document. What would you like to create?")
        return {"messages": [opening_message]}
    else:
        user_input = input("\nWhat would you like to do with the document?")
        user_message = HumanMessage(content=user_input)
    
    all_messages = [system_prompt] + list(state['messages']) + [user_message]
    response = model.invoke(all_messages)
    return {"messages": [user_message, response]}

def should_continue(state: AgentState) -> Literal['continue', 'end']: 
    """Determine if we should continue or end the conversation"""
    messages = state['messages']
    
    if not state['messages']: 
        return 'continue'
    # This looks for the most recent tool message....
    for message in reversed(messages):
        if isinstance(message, ToolMessage):
            content = message.content if isinstance(message.content, str) else str(message.content)
            if "saved" in content.lower() and "document" in content.lower():
                return "end"
        
    return "continue"

In [12]:
def print_messages(messages):
    """Function I made to print the messages in a more readable format"""
    if not messages:
        return
    
    for message in messages[-3:]:
        if isinstance(message, ToolMessage):
            print(f"\n🛠️ TOOL RESULT: {message.content}")

In [13]:
from langgraph.graph import END, START, StateGraph
from langgraph.prebuilt import ToolNode


graph = StateGraph(AgentState)
graph.add_node('agent', agent)
graph.add_node('tools', ToolNode(tools))

graph.add_edge(START, 'agent')
graph.add_edge('agent', 'tools')

graph.add_conditional_edges(
    'tools',
    should_continue,
    {
        'continue': 'agent',
        'end': END
    }
)

app = graph.compile()

In [ ]:
from typing import cast


def run_agent():
    print("\n ===== DRAFTER =====")
    
    state: AgentState = cast(AgentState,{"messages": [], "document": ""})
    
    for step in app.stream(state, stream_mode="values"):
        if "messages" in step:
            print_messages(step["messages"])
    
    print("\n ===== DRAFTER FINISHED =====")

if __name__ == "__main__":
    run_agent() 